Cell 1: Environment Initialization & Strategic Fund Configurations

This cell establishes global parameters. Separating these parameters into their own cell allows adjustment of variables—like the 15% industry cap or the fee structures—without needing to re-run any analytical steps.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import sqlite3
import numpy as np
import pandas as pd
from pathlib import Path

# =====================================================================
# 1. DYNAMIC DATA WAREHOUSE PATH ANCHORING
# =====================================================================
notebook_path = Path(os.getcwd())
root_dir = notebook_path

while root_dir.name != "data_warehouse" and root_dir.parent != root_dir:
    root_dir = root_dir.parent

if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

print(f"🎉 Dynamic Warehouse Path Verified. Anchor Lock: {root_dir}")

# =====================================================================
# 2. CALIBRATED STRATEGIC FUND PROPERTIES & GOVERNANCE GAURDRAILS
# =====================================================================
TOTAL_FUND_CORPUS = 15_000_000        # $15 Million Launch Allocation
SUBSCRIPTION_RATE = 1.0               # Strict 1:1 capacity ceiling limit
SINGLE_INDUSTRY_CAP_PCT = 0.15        # 15% single-industry concentration rule
AVG_LOAN_SIZE = 250_000               # Baseline underlying agribusiness loan size
FUND_FIRST_LOSS_CAP_PCT = 0.20        # Fund covers up to 20% first-loss cushion

# --- FIXED REAL-WORLD TIME BOUNDS ---
WINDOW_MONTHS = 60
ANNUAL_CHECKPOINTS = np.arange(12, WINDOW_MONTHS + 1, 12)  # <-- RESTORED VARIABLE

# --- REAL-WORLD EXPOSURE & AMORTIZATION VARIABLES ---
AVG_SUBSCRIPTION_UTILIZATION_PCT = 0.75  # 15% avg requested coverage vs 20% max
EST_ANNUAL_AMORTIZATION_PAYDOWN = 0.12   # 12% baseline principal paydown per year

# --- REVENUE SUBSIDY FLOOR VARIABLE ---
CORPUS_INTEREST_RATE = 0.0250         # Conservative 2.5% yield on liquid deposits

# --- GLOBAL PORTFOLIO ALLOCATION CAP CONTROLLER ---
SHORT_CLASS_ALLOCATION_CAP = 0.30     # Capped at 30% max face value for short-duration

# Proposed Flat Fee Structures (Assessed directly on Subscription Value Layer)
SHORT_TERM_FEE_RATE = 0.050           # Your original proposed 5.0%
LONG_TERM_FEE_RATE = 0.035            # Your original proposed 3.5%

# Dynamic Underwriting Derivations
CEILING_EXPOSURE_PER_LOAN = AVG_LOAN_SIZE * FUND_FIRST_LOSS_CAP_PCT  # $50,000
AVG_SUBSCRIPTION_EXPOSURE = CEILING_EXPOSURE_PER_LOAN * AVG_SUBSCRIPTION_UTILIZATION_PCT  # $37,500

MAX_TOTAL_FUND_EXPOSURE = TOTAL_FUND_CORPUS * SUBSCRIPTION_RATE  # $15,000,000
MAX_RISK_PER_INDUSTRY = MAX_TOTAL_FUND_EXPOSURE * SINGLE_INDUSTRY_CAP_PCT  # $2,250,000
MAX_ACTIVE_SUBS_PER_INDUSTRY = int(MAX_RISK_PER_INDUSTRY // AVG_SUBSCRIPTION_EXPOSURE)

print("\n--- Calibrated Governance Parameters Enforced ---")
print(f" • Total Core Corpus Capital     : ${TOTAL_FUND_CORPUS:,.2f}")
print(f" • Expected Avg Subscription Size: ${AVG_SUBSCRIPTION_EXPOSURE:,.2f} ({AVG_SUBSCRIPTION_UTILIZATION_PCT:.0%} Utilization)")
print(f" • Conservative Interest Floor   : {CORPUS_INTEREST_RATE:.2%} (${TOTAL_FUND_CORPUS * CORPUS_INTEREST_RATE:,.2f})")
print(f" • Global Short-Duration Pool Cap: {SHORT_CLASS_ALLOCATION_CAP:.0%} (${MAX_TOTAL_FUND_EXPOSURE * SHORT_CLASS_ALLOCATION_CAP:,.2f} Max)")


🎉 Dynamic Warehouse Path Verified. Anchor Lock: /Users/bonwier/PythonProjects/data_warehouse

--- Calibrated Governance Parameters Enforced ---
 • Total Core Corpus Capital     : $15,000,000.00
 • Expected Avg Subscription Size: $37,500.00 (75% Utilization)
 • Conservative Interest Floor   : 2.50% ($375,000.00)
 • Global Short-Duration Pool Cap: 30% ($4,500,000.00 Max)


Cell 2: Transitory Database Connection Layer

This cell handles data ingestion from the saved database. By decoupling data loading from data modeling, workspace remains fast and efficient.

In [12]:
# =====================================================================
# TRANSITORY WAREHOUSE DATA INGESTION
# =====================================================================
# Construct the absolute path natively using Path object division operator
db_path = root_dir / "databases" / "transitory" / "peri_urban_ag_analysis.db"

if not db_path.exists():
    raise FileNotFoundError(
        f"Missing mandatory data asset at verified path: {db_path}\n"
        f"Please run Notebook 1 serialization cells first to generate this transitory layer."
    )

conn = sqlite3.connect(db_path)
df_loans = pd.read_sql_query("SELECT * FROM source_loans_snapshot", conn)
conn.close()

print(f"🎉 Connection Established via Pathlib.")
print(f" • Target Database: {db_path.name}")
print(f" • Micro-Records Ingested to RAM: {len(df_loans):,}")


🎉 Connection Established via Pathlib.
 • Target Database: peri_urban_ag_analysis.db
 • Micro-Records Ingested to RAM: 91,169


Cell 3: Actuarial Survival Vector Algorithms

This cell compiles the mathematical foundation of the model: the corrected risk-pool Kaplan-Meier calculations and the marginal probability of default conversion. Separating this engine makes it highly re-usable for alternative data slices.

In [13]:
# =====================================================================
# LIFE-TABLE MATHEMATICAL ENGINES
# =====================================================================
def extract_actuarial_survival_vector(group_df):
    """
    Computes a mathematically precise Kaplan-Meier cumulative default trajectory,
    properly accounting for monthly risk-pool censoring.
    """
    if len(group_df) == 0:
        return np.zeros(len(ANNUAL_CHECKPOINTS))
        
    monthly_stats = group_df.groupby('survival_months').agg(
        d_i=('event_occurred', 'sum'),
        total_exits=('survival_months', 'count')
    ).sort_index()
    
    total_records = len(group_df)
    cumulative_exits_prior = monthly_stats['total_exits'].cumsum().shift(1).fillna(0)
    monthly_stats['n_i'] = total_records - cumulative_exits_prior
    
    # Calculate step hazard safely
    monthly_stats['step_survival'] = np.where(
        monthly_stats['n_i'] > 0,
        1.0 - (monthly_stats['d_i'] / monthly_stats['n_i']),
        1.0
    )
    monthly_stats['cumulative_survival'] = monthly_stats['step_survival'].cumprod()
    
    # Map back to annual milestones
    cumulative_defaults = []
    for month in ANNUAL_CHECKPOINTS:
        historical_steps = monthly_stats[monthly_stats.index <= month]
        if len(historical_steps) > 0:
            latest_survival = historical_steps['cumulative_survival'].iloc[-1]
        else:
            latest_survival = 1.0
        cumulative_defaults.append(1.0 - latest_survival)
        
    return np.array(cumulative_defaults)

def calculate_peak_marginal_pd(cumulative_defaults):
    """
    Converts a cumulative curve [F_1, F_2, F_3, F_4, F_5] into the 
    highest conditional annual probability of default (q_t).
    """
    F = np.insert(cumulative_defaults, 0, 0.0)
    marginal_pds = []
    
    for t in range(1, len(F)):
        s_prev = 1.0 - F[t-1]
        q_t = (F[t] - F[t-1]) / s_prev if s_prev > 0 else 0.0
        marginal_pds.append(q_t)
        
    return max(marginal_pds) if len(marginal_pds) > 0 else 0.0

print("Actuarial Life-Table algorithms compiled successfully.")


Actuarial Life-Table algorithms compiled successfully.


Cell 4: Routing Engine & Sparse Data Management Strategy

This is the Sparse Policy Controller. By establishing SPARSE_STRATEGY at the top of the cell, we can visually show how switching from PARENT_HIERARCHY to STRIP or PENALTY_BOX alters your asset coverage mapping.

In [14]:
# =====================================================================
# SPARSE DATA CONTROLLER INTERCEPTOR
# Options: 'PARENT_HIERARCHY' (Recommended), 'PENALTY_BOX', 'STRIP'
# =====================================================================
SPARSE_STRATEGY = 'PARENT_HIERARCHY' 

pricing_registry = []
df_targets_only = df_loans[df_loans['is_core_sample'] == 1]
unique_naics_targets = df_targets_only['naics_4d'].unique()

logger_counts = {'NATIVE_CELL': 0, 'PARENT_HIERARCHY': 0, 'PENALTY_BOX_FLOOR': 0, 'STRIP': 0}

for naics in unique_naics_targets:
    for is_long in [0, 1]:
        native_cell = df_targets_only[(df_targets_only['naics_4d'] == naics) & (df_targets_only['is_long_duration'] == is_long)]
        is_sparse = len(native_cell) < 20 or (native_cell['is_sparse_cell'].iloc[0] == 1 if len(native_cell) > 0 else True)
        
        routing_strategy = "NATIVE_CELL"
        active_data = native_cell
        
        if is_sparse:
            if SPARSE_STRATEGY == 'STRIP':
                routing_strategy = "STRIP"
                active_data = pd.DataFrame()
            elif SPARSE_STRATEGY == 'PARENT_HIERARCHY':
                parent_3d = naics[:3]
                active_data = df_loans[(df_loans['naics_3d'] == parent_3d) & (df_loans['is_long_duration'] == is_long)]
                routing_strategy = "PARENT_HIERARCHY" if len(active_data) >= 20 else "PENALTY_BOX_FLOOR"
            elif SPARSE_STRATEGY == 'PENALTY_BOX':
                routing_strategy = "PENALTY_BOX_FLOOR"
                active_data = pd.DataFrame()

        if routing_strategy == "STRIP":
            logger_counts['STRIP'] += 1
            continue
        elif routing_strategy == "PENALTY_BOX_FLOOR" or len(active_data) == 0:
            logger_counts['PENALTY_BOX_FLOOR'] += 1
            peak_empirical_q = 0.065 # 6.5% conservative floor rate
            sample_size = len(native_cell)
        else:
            logger_counts[routing_strategy] += 1
            cum_curve = extract_actuarial_survival_vector(active_data)
            peak_empirical_q = calculate_peak_marginal_pd(cum_curve)
            sample_size = len(active_data)
            
        pricing_registry.append({
            'naics_4d': naics,
            'is_long_duration': is_long,
            'routing_strategy': routing_strategy,
            'sample_size_used': sample_size,
            'peak_annual_pd': peak_empirical_q
        })

df_pricing_matrix = pd.DataFrame(pricing_registry)

# System Summary Metrics
mean_short_pd = df_pricing_matrix[df_pricing_matrix['is_long_duration'] == 0]['peak_annual_pd'].mean()
mean_long_pd = df_pricing_matrix[df_pricing_matrix['is_long_duration'] == 1]['peak_annual_pd'].mean()

print(f"=== DYNAMIC SPARSE TREATMENT REPORT (Policy: '{SPARSE_STRATEGY}') ===")
print(f" • Native Segments Map Securely    : {logger_counts['NATIVE_CELL']}")
print(f" • Parent Hierarchy Intercepts Used: {logger_counts['PARENT_HIERARCHY']}")
print(f" • Penalty Box Floor Rates Imposed : {logger_counts['PENALTY_BOX_FLOOR']}")
print(f" • Segments Stripped From Eligibility: {logger_counts['STRIP']}")
print(f"\n--- Baseline Calculated Empirical Averages ---")
print(f" • Average Empirical Peak PD (Short): {mean_short_pd:.2%} | Target Fee: {SHORT_TERM_FEE_RATE:.2%}")
print(f" • Average Empirical Peak PD (Long) : {mean_long_pd:.2%} | Target Fee: {LONG_TERM_FEE_RATE:.2%}")


=== DYNAMIC SPARSE TREATMENT REPORT (Policy: 'PARENT_HIERARCHY') ===
 • Native Segments Map Securely    : 47
 • Parent Hierarchy Intercepts Used: 3
 • Penalty Box Floor Rates Imposed : 0
 • Segments Stripped From Eligibility: 0

--- Baseline Calculated Empirical Averages ---
 • Average Empirical Peak PD (Short): 22.12% | Target Fee: 5.00%
 • Average Empirical Peak PD (Long) : 1.66% | Target Fee: 3.50%


Cell 5: Constrained Capacity Pipeline & Monte Carlo Stress Testing

This final cell evaluates portfolio cash flows. It generates incoming subscription volume, applies variable concentration limits, subjects the remaining portfolio to systemic stress, and maps the 95% and 99% Value-at-Risk metrics.

In [15]:
# =====================================================================
# CELL 5: CORRECTED CAPACITY-CONSTRAINED MONTE CARLO SIMULATION ENGINE
# =====================================================================
def run_corrected_government_stress_test(df_pricing, n_simulations=10000):
    np.random.seed(42)
    available_naics = df_pricing['naics_4d'].unique()
    
    # Simulate a deep incoming request pipeline buffer to fill out capital limits
    sim_pipeline_buffer = 2000
    pipeline_naics = np.random.choice(available_naics, size=sim_pipeline_buffer)
    
    industry_active_counts = {naics: 0 for naics in available_naics}
    accepted_portfolio = []
    
    short_exposure_running = 0
    total_exposure_running = 0
    blocked_by_industry_cap = 0
    
    # 1. Build the capacity-constrained blended portfolio
    for naics in pipeline_naics:
        if total_exposure_running >= MAX_TOTAL_FUND_EXPOSURE:
            break
            
        is_long = np.random.choice([0, 1], p=[0.40, 0.60])
        
        # --- FIXED ROUTING ENGINE INTERVENTION ---
        # If short-duration is full, drop this entry and fetch a native long-duration asset next
        if is_long == 0 and short_exposure_running >= (MAX_TOTAL_FUND_EXPOSURE * SHORT_CLASS_ALLOCATION_CAP):
            continue  
            
        # Enforce your 15% individual single-industry concentration rule
        if industry_active_counts[naics] < MAX_ACTIVE_SUBS_PER_INDUSTRY:
            match = df_pricing[(df_pricing['naics_4d'] == naics) & (df_pricing['is_long_duration'] == is_long)]
            
            # Safeguard against indexing errors on sparse/missing cells
            base_pd = match['peak_annual_pd'].values[0] if len(match) > 0 else 0.04
            
            # Premium fee collected strictly on the expected subscription layer ($37,500)
            fee_collected = AVG_SUBSCRIPTION_EXPOSURE * (SHORT_TERM_FEE_RATE if is_long == 0 else LONG_TERM_FEE_RATE)
            
            industry_active_counts[naics] += 1
            total_exposure_running += AVG_SUBSCRIPTION_EXPOSURE
            if is_long == 0:
                short_exposure_running += AVG_SUBSCRIPTION_EXPOSURE
                
            accepted_portfolio.append({
                'naics_4d': naics,
                'is_long_duration': is_long,
                'fee_revenue': fee_collected,
                'baseline_annual_pd': base_pd
            })
        else:
            blocked_by_industry_cap += 1
            
    df_active_fund = pd.DataFrame(accepted_portfolio)
    total_premium_revenue = df_active_fund['fee_revenue'].sum()
    total_active_loans = len(df_active_fund)
    n_short = len(df_active_fund[df_active_fund['is_long_duration'] == 0])
    n_long = len(df_active_fund[df_active_fund['is_long_duration'] == 1])
    
    # 2. Map baseline default trials directly against raw SBA history (Stress Multiplier = 1.0)
    pds_baseline = np.clip(df_active_fund['baseline_annual_pd'].to_numpy() * 1.0, 0.0, 0.95)
    
    # Execute 10,000 independent parallel distribution trials
    random_matrix = np.random.rand(n_simulations, total_active_loans)
    default_matrix = random_matrix < pds_baseline
    
    # 3. Process Amortization Paydowns across simulations based on default year tracking
    sim_default_years = np.random.randint(1, 6, size=(n_simulations, total_active_loans))
    paydown_multipliers = 1.0 - ((sim_default_years - 1) * EST_ANNUAL_AMORTIZATION_PAYDOWN)
    paydown_multipliers = np.clip(paydown_multipliers, 0.20, 1.0) # Establish baseline recovery floor
    
    # Calculate claims paid out on the paid-down average subscription balances
    simulated_claims = (default_matrix * paydown_multipliers).sum(axis=1) * AVG_SUBSCRIPTION_EXPOSURE
    
    # 4. Integrate Non-Premium Income Buffer (2.5% Conservative Treasury Yield Floor)
    guaranteed_interest_income = TOTAL_FUND_CORPUS * CORPUS_INTEREST_RATE
    total_annual_revenue_inflow = total_premium_revenue + guaranteed_interest_income
    
    # Solvency Analytics
    ending_corpus_positions = TOTAL_FUND_CORPUS + total_annual_revenue_inflow - simulated_claims
    insolvency_events = np.sum(ending_corpus_positions < 0)
    var_95 = np.percentile(ending_corpus_positions, 5)
    var_99 = np.percentile(ending_corpus_positions, 1)
    
    # =====================================================================
    # REVISED EXECUTIVE CALIBRATION STRESS REPORT
    # =====================================================================
    print("\n" + "="*65)
    print("      PERI-URBAN COLLATERAL BRIDGE FUND: CORPUS STRESS REPORT")
    print("="*65)
    print(f" INITIAL GOVERNMENT CORPUS BASE ALLOC : ${TOTAL_FUND_CORPUS:,.2f}")
    print(f" CORPUS CONSERVATIVE INTEREST FLOOR : {CORPUS_INTEREST_RATE:.2%} (${guaranteed_interest_income:,.2f})")
    print(f" TARGET SUBSCRIPTION CAPACITY MULT   : {SUBSCRIPTION_RATE:.2f}x Capacity Limit")
    print(f" SINGLE-INDUSTRY CONCENTRATION LIMIT  : {SINGLE_INDUSTRY_CAP_PCT:.1%} (${MAX_RISK_PER_INDUSTRY:,.2f})")
    print(f" -------------------------------------------------------------")
    print(f" ACTIVE SUBSCRIPTIONS SAFELY DEPLOYED : {total_active_loans} loans")
    print(f"  --> Short-Duration Working Cap (30%): {n_short} loans (${n_short * AVG_SUBSCRIPTION_EXPOSURE:,.2f})")
    print(f"  --> Long-Duration Infrastructure     : {n_long} loans (${n_long * AVG_SUBSCRIPTION_EXPOSURE:,.2f})")
    print(f" ISSUANCES BLOCKED VIA INDUSTRY CAP   : {blocked_by_industry_cap} applications")
    print(f" TRUE FUND TOTAL LIABILITY ISSUED     : ${total_active_loans * AVG_SUBSCRIPTION_EXPOSURE:,.2f}")
    print(f" -------------------------------------------------------------")
    print(f" TOTAL UPFRONT FEE PREMIUMS COLLECTED : ${total_premium_revenue:,.2f}")
    print(f" TOTAL COMPREHENSIVE REVENUE INFLOW  : ${total_annual_revenue_inflow:,.2f}")
    print(f" EXPECTED CLAIMS PAYOUT (SBA BASELINE): ${simulated_claims.mean():,.2f}")
    print(f" PROJECTED YEAR-1 OPERATIONAL CASH FLOW: ${total_annual_revenue_inflow - simulated_claims.mean():,.2f}")
    print(f" -------------------------------------------------------------")
    print(f" VALUE-AT-RISK (95% CONFIDENCE CORPUS): ${max(var_95, 0):,.2f}")
    print(f" VALUE-AT-RISK (99% CONFIDENCE CORPUS): ${max(var_99, 0):,.2f}")
    print(f" PROBABILITY OF FUND CAPITAL DEPLETION : {insolvency_events / n_simulations:.4%}")
    print("="*65)

run_corrected_government_stress_test(df_pricing_matrix)



      PERI-URBAN COLLATERAL BRIDGE FUND: CORPUS STRESS REPORT
 INITIAL GOVERNMENT CORPUS BASE ALLOC : $15,000,000.00
 CORPUS CONSERVATIVE INTEREST FLOOR : 2.50% ($375,000.00)
 TARGET SUBSCRIPTION CAPACITY MULT   : 1.00x Capacity Limit
 SINGLE-INDUSTRY CONCENTRATION LIMIT  : 15.0% ($2,250,000.00)
 -------------------------------------------------------------
 ACTIVE SUBSCRIPTIONS SAFELY DEPLOYED : 400 loans
  --> Short-Duration Working Cap (30%): 120 loans ($4,500,000.00)
  --> Long-Duration Infrastructure     : 280 loans ($10,500,000.00)
 ISSUANCES BLOCKED VIA INDUSTRY CAP   : 0 applications
 TRUE FUND TOTAL LIABILITY ISSUED     : $15,000,000.00
 -------------------------------------------------------------
 TOTAL UPFRONT FEE PREMIUMS COLLECTED : $592,500.00
 TOTAL COMPREHENSIVE REVENUE INFLOW  : $967,500.00
 EXPECTED CLAIMS PAYOUT (SBA BASELINE): $873,589.35
 PROJECTED YEAR-1 OPERATIONAL CASH FLOW: $93,910.65
 -------------------------------------------------------------
 VALUE-AT-RI